In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"                                              

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant")

tools = [read_email_tool, send_email_tool]

agent  = create_agent(
    model = model,
    tools = tools,
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [8]:
from langchain_core.messages import HumanMessage

config = {"configurable":{"thread_id":"test-approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to smith@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to smith@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7dbce2ff-fc50-4b56-924d-663ac7743e04'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'n5vyj19cf', 'function': {'arguments': '{"body":"How are you?","recipient":"smith@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.029165201, 'completion_tokens_details': None, 'prompt_time': 0.020827981, 'prompt_tokens_details': None, 'queue_time': 0.046932889, 'total_time': 0.049993182}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d00c5-f522-70a2-94a4-3f6679c3c7d4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

In [10]:
from langgraph.types import Command

## Step 2: Approve
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: This will send the same email again. If you want to avoid this, you should add error checking after the send email call to prevent duplicate emails from being sent.


In [11]:
result

{'messages': [HumanMessage(content="Send email to smith@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7dbce2ff-fc50-4b56-924d-663ac7743e04'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'n5vyj19cf', 'function': {'arguments': '{"body":"How are you?","recipient":"smith@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.029165201, 'completion_tokens_details': None, 'prompt_time': 0.020827981, 'prompt_tokens_details': None, 'queue_time': 0.046932889, 'total_time': 0.049993182}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d00c5-f522-70a2-94a4-3f6679c3c7d4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo